# End-to-End Icechunk Test\n\nWrite 3 FVCOM files to a temp Icechunk store, read back, plot surface temperature.\nUses `s3://umassd-fvcom/gom3/hindcast/icechunk/test3.icechunk` (overwritten each run).

In [1]:
import json, os, struct, time
import cftime
import numpy as np
import s3fs
import xarray as xr
import icechunk
from kerchunk.netCDF3 import NetCDF3ToZarr
from kerchunk.utils import subchunk
from obstore.store import S3Store
from obspec_utils.registry import ObjectStoreRegistry
from pathlib import Path
from virtualizarr.manifests import ManifestStore
from virtualizarr.parsers.kerchunk.translator import manifestgroup_from_kerchunk_refs
from dotenv import load_dotenv

load_dotenv(os.path.expanduser('~/dotenv/umassd-fvcom.env'))

True

In [2]:
import icechunk, virtualizarr
print(icechunk.__version__, virtualizarr.__version__)

1.1.21 2.5.0


## Configuration and file list

In [3]:
SKIP_VARS     = ['Itime', 'Itime2', 'Times', 'file_date', 'iint', 'nprocs']
SIGLEV_VARS   = ['kh', 'km', 'kq', 'l', 'omega', 'q2', 'q2l', 'siglev']
SIGLAY_VARS   = ['salinity', 'siglay', 'temp', 'u', 'v', 'ww']
NLEV, NLAY    = 46, 45
bucket, region = 'umassd-fvcom', 'us-east-1'

fs = s3fs.S3FileSystem(anon=True)
flist = sorted(fs.glob('umassd-fvcom/gom3/hindcast/*.nc'))
flist = [f's3://{f}' for f in flist]
url_0 = flist[0]
TEST_FILES = flist[:3]
print(f'Testing with {len(TEST_FILES)} files: {[f.split("/")[-1] for f in TEST_FILES]}')

Testing with 3 files: ['gom3_197801.nc', 'gom3_197802.nc', 'gom3_197803.nc']


## Time metadata and shared S3 registry

In [4]:
# Read time metadata from first file to construct uniform time coordinates
_refs_0   = NetCDF3ToZarr(url_0, inline_threshold=0, storage_options={'anon': True}).translate()['refs']
time_zattrs   = json.loads(_refs_0['time/.zattrs'])
time_units    = time_zattrs['units']
time_calendar = time_zattrs.get('calendar', 'standard')
_be_dtype     = np.dtype(json.loads(_refs_0['time/.zarray'])['dtype']).newbyteorder('>')
ref_t0, ref_t1 = _refs_0['time/0'], _refs_0['time/1']
with fs.open(url_0, 'rb') as _f:
    _f.seek(ref_t0[1]); t0_val = np.frombuffer(_f.read(ref_t0[2]), dtype=_be_dtype)[0]
    _f.seek(ref_t1[1]); t1_val = np.frombuffer(_f.read(ref_t1[2]), dtype=_be_dtype)[0]
dt_val = t1_val - t0_val

# Shared S3 registry — reused for every file's ManifestStore
_s3_store = S3Store.from_url(f's3://{bucket}', config={'skip_signature': True, 'region': region})
_registry = ObjectStoreRegistry({f's3://{bucket}': _s3_store})

print(f't0={cftime.num2date(t0_val, time_units, time_calendar)}  dt={dt_val*24:.4f}h')

t0=1978-01-01 00:00:00  dt=1.0312h


## Helper functions

In [5]:
def get_ntime_from_header(url, fs):
    with fs.open(url, 'rb') as f:
        hdr = f.read(8)
    assert hdr[:3] == b'CDF'
    return struct.unpack('>i', hdr[4:8])[0]

def make_vds(url, step_count):
    """Build a virtual dataset for one FVCOM file: kerchunk + subchunk + computed time coord."""
    refs = NetCDF3ToZarr(url, inline_threshold=0, storage_options={'anon': True}).translate()['refs']
    for v in SIGLEV_VARS: refs = subchunk(refs, variable=v, factor=NLEV)
    for v in SIGLAY_VARS:  refs = subchunk(refs, variable=v, factor=NLAY)

    ntime     = json.loads(refs['time/.zarray'])['shape'][0]
    refs_data = {k: v for k, v in refs.items() if not k.startswith('time')}

    mg  = manifestgroup_from_kerchunk_refs({'version': 1, 'refs': refs_data},
                                           skip_variables=SKIP_VARS,
                                           fs_root=Path.cwd().as_uri())
    ms  = ManifestStore(group=mg, registry=_registry)
    vds = ms.to_virtual_dataset(loadable_variables=[], decode_times=False)

    raw        = t0_val + (step_count + np.arange(ntime)) * dt_val
    time_coord = xr.Variable('time', cftime.num2date(raw, time_units, time_calendar))
    return xr.Dataset(dict(vds.data_vars), coords={'time': time_coord}, attrs=vds.attrs), ntime

# Per-variable CF attributes (from NcML)
CF_VAR_ATTRS = {
    'time':     {'standard_name': 'time'},
    'h':        {'standard_name': 'sea_floor_depth_below_geoid', 'units': 'm',
                 'coordinates': 'lat lon'},
    'zeta':     {'standard_name': 'sea_surface_height_above_geoid', 'units': 'meters',
                 'coordinates': 'time lat lon', 'coverage_content_type': 'modelResult'},
    'temp':     {'standard_name': 'sea_water_potential_temperature',
                 'coordinates': 'time siglay lat lon', 'coverage_content_type': 'modelResult'},
    'salinity': {'standard_name': 'sea_water_salinity', 'units': '0.001',
                 'coordinates': 'time siglay lat lon', 'coverage_content_type': 'modelResult'},
    'u':        {'standard_name': 'eastward_sea_water_velocity', 'units': 'meters s-1',
                 'coordinates': 'time siglay latc lonc', 'coverage_content_type': 'modelResult'},
    'v':        {'standard_name': 'northward_sea_water_velocity', 'units': 'meters s-1',
                 'coordinates': 'time siglay latc lonc', 'coverage_content_type': 'modelResult'},
    'ww':       {'standard_name': 'upward_sea_water_velocity', 'units': 'meters s-1',
                 'coordinates': 'time siglay latc lonc', 'coverage_content_type': 'modelResult'},
    'ua':       {'standard_name': 'barotropic_eastward_sea_water_velocity', 'units': 'meters s-1',
                 'coordinates': 'time latc lonc', 'coverage_content_type': 'modelResult'},
    'va':       {'standard_name': 'northward_barotropic_sea_water_velocity', 'units': 'meters s-1',
                 'coordinates': 'time latc lonc', 'coverage_content_type': 'modelResult'},
    'siglay':   {'standard_name': 'ocean_sigma_coordinate', 'positive': 'up',
                 'valid_min': -1.0, 'valid_max': 0.0,
                 'formula_terms': 'sigma: siglay eta: zeta depth: h'},
    'nv':       {'long_name': 'nodes surrounding element',
                 'cf_role': 'face_node_connectivity', 'start_index': 1},
}

mesh_topology_ds = xr.Dataset({'mesh_topology': xr.Variable((), np.int32(0), attrs={
    'cf_role': 'mesh_topology',
    'topology_dimension': 2,
    'node_coordinates': 'lon lat',
    'face_coordinates': 'lonc latc',
    'face_node_connectivity': 'nv',
    'face_dimension': 'nele',
})})

def add_ugrid_metadata(ds):
    """Add CF-1.11/UGRID attrs and merge in the mesh_topology scalar variable."""
    ds.attrs['Conventions'] = 'CF-1.11'
    for var, attrs in CF_VAR_ATTRS.items():
        if var in ds:
            ds[var].attrs.update(attrs)
    for var in ds.data_vars:
        dims = ds[var].dims
        if 'node' in dims or 'nele' in dims:
            ds[var].attrs.setdefault('mesh', 'mesh_topology')
            ds[var].attrs.setdefault('location', 'face' if 'nele' in dims else 'node')
    return xr.merge([ds, mesh_topology_ds], combine_attrs='no_conflicts')

## Create Icechunk store (deletes any previous test run)

In [6]:
%%time
test_prefix = 'gom3/hindcast/icechunk/test3.icechunk'
write_fs = s3fs.S3FileSystem(
    key=os.environ['AWS_ACCESS_KEY_ID'],
    secret=os.environ['AWS_SECRET_ACCESS_KEY'],
)
test_path = f'{bucket}/{test_prefix}'
if write_fs.exists(test_path):
    write_fs.rm(test_path, recursive=True)
    print(f'Deleted existing test store at s3://{test_path}')

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix='s3://umassd-fvcom/',
        store=icechunk.s3_store(region=region, anonymous=True),
    ),
)
storage = icechunk.s3_storage(
    bucket=bucket, prefix=test_prefix, region=region,
    access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
)
repo = icechunk.Repository.open_or_create(storage, config)
print('Icechunk repo ready')

Deleted existing test store at s3://umassd-fvcom/gom3/hindcast/icechunk/test3.icechunk
Icechunk repo ready
CPU times: user 272 ms, sys: 53 ms, total: 325 ms
Wall time: 2.47 s


## Write files to Icechunk

In [7]:
%%time
total_start = time.perf_counter()
step_count  = 0
time_var_names = None
session = repo.writable_session('main')

for i, f in enumerate(TEST_FILES):
    t0 = time.perf_counter()
    vds, ntime_n = make_vds(f, step_count)
    step_count += ntime_n

    if i == 0:
        time_var_names = {name for name, var in vds.data_vars.items()
                          if var.dims and var.dims[0] == 'time'}
        vds = add_ugrid_metadata(vds)
        vds.vz.to_icechunk(session.store)
    else:
        # Only append time-varying variables — static vars (nv, lon, lat, etc.) must not
        # be overwritten since their virtual chunk offsets are specific to file 0.
        vds[list(time_var_names)].vz.to_icechunk(session.store, append_dim='time')

    print(f'  [{i+1}/{len(TEST_FILES)}] {f.split("/")[-1]}  ntime={ntime_n}  '
          f'cumulative={step_count}  {time.perf_counter()-t0:.1f}s  '
          f'(elapsed: {time.perf_counter()-total_start:.1f}s)')

snapshot_id = session.commit(f'test: {len(TEST_FILES)} FVCOM files')
print(f'\nDone. Total steps={step_count}  snapshot={snapshot_id}'
      f'\nTotal elapsed: {time.perf_counter()-total_start:.1f}s')

  [1/3] gom3_197801.nc  ntime=745  cumulative=745  46.1s  (elapsed: 46.2s)
  [2/3] gom3_197802.nc  ntime=673  cumulative=1418  46.1s  (elapsed: 92.3s)
  [3/3] gom3_197803.nc  ntime=745  cumulative=2163  62.5s  (elapsed: 154.8s)

Done. Total steps=2163  snapshot=K1QEBSE707P9Y8T4SEJ0
Total elapsed: 165.1s
CPU times: user 2min 18s, sys: 3.85 s, total: 2min 22s
Wall time: 2min 45s


## Read back and validate

In [8]:
test_prefix = 'gom3/hindcast/icechunk/test3.icechunk'

In [9]:
config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix='s3://umassd-fvcom/',
        store=icechunk.s3_store(region=region, anonymous=True),
    ),
)

In [10]:
storage = icechunk.s3_storage(
    bucket=bucket, prefix=test_prefix, region=region,
    access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY'],
)

In [11]:
%%time
credentials = icechunk.containers_credentials({
    's3://umassd-fvcom/': icechunk.s3_credentials(anonymous=True)
})
repo_ro    = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
session_ro = repo_ro.readonly_session('main')
ds = xr.open_zarr(session_ro.store, consolidated=False, chunks={})

CPU times: user 599 ms, sys: 80 ms, total: 679 ms
Wall time: 1.4 s


In [12]:
ds

<xarray.Dataset> Size: 283GB
Dimensions:        (four: 4, nele: 90415, node: 48451, three: 3, time: 2163,
                    siglev: 46, maxnode: 11, maxelem: 9, siglay: 45)
Coordinates:
    latc           (nele) float32 362kB dask.array<chunksize=(90415,), meta=np.ndarray>
    lonc           (nele) float32 362kB dask.array<chunksize=(90415,), meta=np.ndarray>
    lat            (node) float32 194kB dask.array<chunksize=(48451,), meta=np.ndarray>
    lon            (node) float32 194kB dask.array<chunksize=(48451,), meta=np.ndarray>
    x              (node) float32 194kB dask.array<chunksize=(48451,), meta=np.ndarray>
    y              (node) float32 194kB dask.array<chunksize=(48451,), meta=np.ndarray>
  * time           (time) datetime64[ns] 17kB 1978-01-01 ... 1978-04-03T21:33:45
    siglev         (siglev, node) float32 9MB dask.array<chunksize=(1, 48451), meta=np.ndarray>
    siglay         (siglay, node) float32 9MB dask.array<chunksize=(1, 48451), meta=np.ndarray>
Dimensions without coordinates: four, nele, node, three, maxnode, maxelem
Data variables: (12/37)
    a1u            (four, nele) float32 1MB dask.array<chunksize=(4, 90415), meta=np.ndarray>
    art2           (node) float32 194kB dask.array<chunksize=(48451,), meta=np.ndarray>
    art1           (node) float32 194kB dask.array<chunksize=(48451,), meta=np.ndarray>
    awy            (three, nele) float32 1MB dask.array<chunksize=(3, 90415), meta=np.ndarray>
    aw0            (three, nele) float32 1MB dask.array<chunksize=(3, 90415), meta=np.ndarray>
    km             (time, siglev, node) float32 19GB dask.array<chunksize=(1, 1, 48451), meta=np.ndarray>
    ...             ...
    ww             (time, siglay, nele) float32 35GB dask.array<chunksize=(1, 1, 90415), meta=np.ndarray>
    ua             (time, nele) float32 782MB dask.array<chunksize=(1, 90415), meta=np.ndarray>
    vwind_stress   (time, nele) float32 782MB dask.array<chunksize=(1, 90415), meta=np.ndarray>
    xc             (nele) float32 362kB dask.array<chunksize=(90415,), meta=np.ndarray>
    zeta           (time, node) float32 419MB dask.array<chunksize=(1, 48451), meta=np.ndarray>
    yc             (nele) float32 362kB dask.array<chunksize=(90415,), meta=np.ndarray>
Attributes: (12/14)
    title:                       GOM3 1978 Nesting by Yf.Sun@umassd.edu
    institution:                 School for Marine Science and Technology
    source:                      FVCOM_3.0
    history:                     model started at: 22/12/2011   10:08
    references:                  http://fvcom.smast.umassd.edu, http://codfis...
    Conventions:                 CF-1.0
    ...                          ...
    Tidal_Forcing:               TIDAL ELEVATION FORCING IS OFF!
    River_Forcing:               THERE ARE 50 RIVERS IN THIS MODEL.\nRIVER IN...
    GroundWater_Forcing:         GROUND WATER FORCING IS OFF!
    Surface_Heat_Forcing:        FVCOM variable surface heat forcing file:\nF...
    Surface_Wind_Forcing:        FVCOM variable surface Wind forcing:\nFILE N...
    Surface_PrecipEvap_Forcing:  FVCOM periodic surface precip forcing:\nFILE...

In [13]:
import numpy as np
import holoviews as hv
import geoviews as gv
import cartopy.crs as ccrs
import hvplot.xarray
import holoviews.operation.datashader as dshade
import pandas as pd

dshade.datashade.precompute = True
hv.extension('bokeh')

In [14]:
var_name='temp'  # variable
level = 0  # vertical level (0 is surface, -1 is bottom)
da = ds[var_name].isel(siglay=0).isel(time=-1)
time = da.time.values
values = da.load()

In [15]:
v = np.vstack((ds['lon'], ds['lat'], values)).T
verts = pd.DataFrame(v, columns=['lon','lat','var'])

In [16]:
points = gv.operation.project_points(gv.Points(verts, vdims=['var']))

In [17]:
tris = pd.DataFrame(ds['nv'].T.values.astype('int')-1, columns=['v0','v1','v2'])
tiles = gv.tile_sources.OSM
title = f'{var_name}@level {level}   time:{time}'


In [18]:
print(v.shape)
print(verts.shape)
print(points.shape)

(48451, 3)
(48451, 3)
(48451, 3)


In [19]:

trimesh = gv.TriMesh((tris, points), label=title)
mesh = dshade.rasterize(trimesh).opts(
              cmap='rainbow', colorbar=True, width=600, height=400)

tiles * mesh

BokehModel(combine_events=True, render_bundle={'docs_json': {'431b772a-094a-488b-b555-79b016ec5089': {'version…

In [20]:
print(ds.lat.min().values, ds.lat.max().values, ds.lon.min().values, ds.lon.max().values)

35.28387451171875 46.14595413208008 -75.68433380126953 -56.85079574584961
